In [ ]:
import datetime

# Get the current date and time
start_datetime = datetime.datetime.now()

# Print the current date and time in a standard format
print(f"This script started running at: {start_datetime}")

In [1]:
from google.colab import auth

# Authenticate user credentials for google colab
auth.authenticate_user()
print('Authenticated')

Authenticated


In [2]:
from google.cloud import bigquery

# Declare the default bigquery project to use
project = 'ld-pcx-bia'

# Connect to bigquery
client = bigquery.Client(project=project)

In [3]:
import os

current_wd = os.getcwd()

print(f"Current working directory: {os.getcwd()}")

Current working directory: /content


In [4]:
# Create a dataframe called df with the saved results of the query run

df = client.query('''

  SELECT * FROM `ld-pcx-bia.jongrub.IMAGE_SIMILARITY_S3_ASSETFUL`

''' ).to_dataframe()

# Preview that the dataframe looks as expected
print(df.head())

             liam      article  \
0  20812144001_EA  20812144001   
1  20067389001_EA  20067389001   
2     21341017_EA     21341017   
3     21290737_EA     21290737   
4  20811994001_EA  20811994001   

                                              s3_url  \
0  https://clickncollect.s3.amazonaws.com/product...   
1  https://clickncollect.s3.amazonaws.com/product...   
2  https://clickncollect.s3.amazonaws.com/product...   
3  https://clickncollect.s3.amazonaws.com/product...   
4  https://clickncollect.s3.amazonaws.com/product...   

                                        assetful_url  
0  https://digital.loblaws.ca/PCX/20812144001_EA/...  
1  https://digital.loblaws.ca/PCX/20067389001_EA/...  
2  https://digital.loblaws.ca/PCX/21341017_EA/en/...  
3  https://digital.loblaws.ca/PCX/21290737_EA/en/...  
4  https://digital.loblaws.ca/PCX/20811994001_EA/...  


In [5]:
# If you have not installed the required packages, remove the '#' from the line below so it just says '!pip install pandas requests PIL imagehash io' and run again
#pip install pandas requests PIL imagehash io
!pip install pandas requests PIL imagehash io
!pip install imagehash
import pandas as pd
import requests
from PIL import Image
import imagehash
from io import BytesIO

# --- Configuration (Column Names) ---
# These constants define the expected column names in your input DataFrame
S3_URL_COLUMN = 's3_url'
ASSETFUL_URL_COLUMN = 'assetful_url'
LIAM_COLUMN = 'LIAM' # Used for logging progress

# --- Helper Function ---

def download_and_hash_image(url):
    """
    Downloads an image from a URL and returns its perceptual hash (phash).
    Returns None if the download or processing fails.
    """
    if pd.isna(url) or not isinstance(url, str) or not url.startswith('http'):
        # print(f"  Skipping invalid or missing URL: {url}") # Uncomment for verbose skipping
        return None

    try:
        # Download the image with a timeout
        response = requests.get(url, stream=True, timeout=10)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

        # Check if the content type is likely an image (optional, but good practice)
        content_type = response.headers.get('Content-Type', '')
        if not content_type.startswith('image/'):
            print(f"  Warning: URL {url} did not return an image content type: {content_type}")

        # Open the image from bytes in memory using PIL
        image_data = BytesIO(response.content)
        img = Image.open(image_data)

        # Convert image to RGB mode to ensure compatibility with imagehash and avoid errors
        if img.mode != 'RGB':
            img = img.convert('RGB')

        # Calculate the perceptual hash (phash)
        img_hash = imagehash.phash(img)
        return img_hash
    except requests.exceptions.RequestException as e:
        print(f"  Error downloading {url}: {e}")
        return None
    except Image.UnidentifiedImageError:
        print(f"  Error: Could not identify image from {url}. It might not be a valid image file.")
        return None
    except Exception as e:
        print(f"  An unexpected error occurred processing {url}: {e}")
        return None

# --- Main Processing Function ---

def calculate_image_similarity_for_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates the perceptual similarity score between images from two URL columns
    in a DataFrame and adds the score as a new column.

    Args:
        input_df (pd.DataFrame): The input DataFrame containing 's3_url' and
                                 'assetful_url' columns (and optionally 'LIAM').

    Returns:
        pd.DataFrame: The DataFrame with an added 'similarity_score' column.
                      A lower score indicates higher visual similarity.
    """
    if not all(col in df.columns for col in [S3_URL_COLUMN, ASSETFUL_URL_COLUMN]):
        raise ValueError(
            f"Input DataFrame must contain '{S3_URL_COLUMN}' and '{ASSETFUL_URL_COLUMN}' columns."
        )

    # Create a copy to avoid modifying the original DataFrame directly
    df_copy = df.copy()

    # Initialize a new column to store the similarity scores
    df['similarity_score'] = None

    print("\nStarting image download and similarity calculation...")
    for index, row in df.iterrows():
        # Get an identifier for logging purposes
        liam_id = row[LIAM_COLUMN] if LIAM_COLUMN in df.columns else f"Row {index + 1}"

        s3_url = row.get(S3_URL_COLUMN)
        assetful_url = row.get(ASSETFUL_URL_COLUMN)

        print(f"\nProcessing {liam_id} ({index + 1}/{len(df)}):")
        print(f"  S3 URL: {s3_url}")
        print(f"  Assetful URL: {assetful_url}")

        # Download and hash the first image
        hash1 = download_and_hash_image(s3_url)
        # Download and hash the second image
        hash2 = download_and_hash_image(assetful_url)

        score = None
        if hash1 is not None and hash2 is not None:
            # Calculate the Hamming distance between the two hashes
            # A score of 0 means the images are perceptually identical
            # Higher scores mean less similarity
            score = hash1 - hash2
            print(f"  Calculated Similarity Score (Hamming Distance): {score}")
        else:
            print("  Could not calculate score for this pair due to errors.")

        # Store the calculated score in the DataFrame
        df.at[index, 'similarity_score'] = score

    print("\nImage processing complete.")
    return df

# --- Example Usage (How to use this function) ---
if __name__ == '__main__':
    # 1. Create a sample DataFrame (or load from an existing source)
    print("--- Example Usage ---")
    data = {
        'LIAM': ['20252014_example_1', '20252014_example_2', '20252014_example_3'],
        's3_url': [
            'https://digital.loblaws.ca/PCX/20252014_EA/en/1/20252014_en_front_1200.png',
            'https://digital.loblaws.ca/PCX/20252014_EA/en/1/20252014_en_front_1200.png',
            'https://digital.loblaws.ca/PCX/20252014_EA/en/3/20252014_en_side_1200.png'
        ],
        'assetful_url': [
            'https://digital.loblaws.ca/PCX/20252014_EA/en/3/20252014_en_side_1200.png',
            'https://digital.loblaws.ca/PCX/20252014_EA/en/7/20252014_en_closed_1200.png',
            'https://digital.loblaws.ca/PCX/20252014_EA/en/7/20252014_en_closed_1200.png'
        ]
    }
    sample_df = pd.DataFrame(data)
    print("\nInitial DataFrame:")
    print(sample_df)

    # 2. Call the function with your DataFrame
    processed_df = calculate_image_similarity_for_dataframe(df)

    # 3. View the processed DataFrame
    print("\nProcessed DataFrame with Similarity Scores:")
    print(processed_df)

    # 4. (Optional) Save the result to a new CSV file
    output_csv_file = 'output_images_with_scores_from_df.csv'
    print(f"\nSaving the processed DataFrame to {output_csv_file}...")
    processed_df.to_csv(output_csv_file, index=False)
    print("Done! Check your directory for the output CSV.")
    print("Remember: A lower 'similarity_score' (Hamming distance) indicates higher visual similarity.")
    print("A score of 0 means the images are perceptually identical according to phash.")

Streaming output truncated to the last 5000 lines.
  Calculated Similarity Score (Hamming Distance): 0

Processing Row 14058 (14058/14882):
  S3 URL: https://clickncollect.s3.amazonaws.com/products/21654374/b1/en/front/21654374_front_a05.png
  Assetful URL: https://digital.loblaws.ca/PCX/21654374_C16/en/1/21654374_enfr_front_1200.png
  Calculated Similarity Score (Hamming Distance): 0

Processing Row 14059 (14059/14882):
  S3 URL: https://clickncollect.s3.amazonaws.com/products/20867041/b1/en/front/20867041_front_a05.png
  Assetful URL: https://digital.loblaws.ca/PCX/20867041_EA/en/1/20867041_en_front_1200.png
  Calculated Similarity Score (Hamming Distance): 0

Processing Row 14060 (14060/14882):
  S3 URL: https://clickncollect.s3.amazonaws.com/products/20031471/b1/en/front/20031471_front_a05.png
  Assetful URL: https://digital.loblaws.ca/PCX/20031471_C06/en/1/20031471_en_front_1200.png
  Calculated Similarity Score (Hamming Distance): 2

Processing Row 14061 (14061/14882):
  S3 URL: 

In [5]:
import datetime

# Get the current date and time
end_datetime = datetime.datetime.now()

# Print the current date and time in a standard format
print(f"This script completed running at: {end_datetime}")